In [1]:
#Setup, Dapta splitting, and setting a naive baseline 

import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import os


os.makedirs('../models', exist_ok=True)

# Load Data
df = pd.read_csv("../data/processed/featured_dataset.csv", index_col=0, parse_dates=True)
df = df.sort_index()

# Defininf target and features
target_col = 'Max. Demand at eve. peak (Generation end)'
X = df.drop(columns=[target_col] + [col for col in df.columns if 'load' in col.lower()])
y = df[target_col]

# Time-Based Validation Split as per sir's feedback and setting aside last 20% for testing set 
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training instances: {len(X_train)} | Test instances: {len(X_test)}")

# Trivial Baseline Model
baseline_preds = X_test['Demand_Lag1']
baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_preds))

print("\n--- Baseline Performance (Naive Persistence) ---")
print(f"Baseline MAE: {baseline_mae:.2f} MW")
print(f"Baseline RMSE: {baseline_rmse:.2f} MW")

Training instances: 1447 | Test instances: 362

--- Baseline Performance (Naive Persistence) ---
Baseline MAE: 1224.30 MW
Baseline RMSE: 1793.16 MW


In [2]:
# Model 01- Regularized Linear model and tuning, we areusing the TimeSeriesSplit for the cross validation

# TimeSeriesSplit for rolling validation 
tscv = TimeSeriesSplit(n_splits=5)

# Ridge Regression Pipeline with scaling 
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge())
])

# Hyperparameter Search Space 
ridge_param_grid = {
    'ridge__alpha': [0.1, 1.0, 10.0, 50.0, 100.0]
}

# Grid Search setup
ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

# Train and Tune
print("Training and tuning Ridge Regression...")
ridge_grid.fit(X_train, y_train)

print(f"Best Ridge Parameters: {ridge_grid.best_params_}")
print(f"Best Ridge Cross-Validation MAE: {-ridge_grid.best_score_:.2f} MW")
joblib.dump(ridge_grid.best_estimator_, '../models/best_ridge_model.pkl')

Training and tuning Ridge Regression...
Best Ridge Parameters: {'ridge__alpha': 0.1}
Best Ridge Cross-Validation MAE: 228.41 MW


['../models/best_ridge_model.pkl']

In [3]:
#Model-02, Random Forest 

# Random Forest Regressor
rf = RandomForestRegressor(random_state=42)

# Hyperparameter Search Space (my pc trash, hence smaller searchspace)
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

# Grid Search setup
rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

# Train and Tune
print("Training and tuning Random Forest (this may take a minute)...")
rf_grid.fit(X_train, y_train)

print(f"Best Random Forest Parameters: {rf_grid.best_params_}")
print(f"Best RF Cross-Validation MAE: {-rf_grid.best_score_:.2f} MW")


joblib.dump(rf_grid.best_estimator_, '../models/best_rf_model.pkl')

Training and tuning Random Forest (this may take a minute)...
Best Random Forest Parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}
Best RF Cross-Validation MAE: 302.99 MW


['../models/best_rf_model.pkl']

In [ ]:
#Shihab, Uporer Shob Cell Run dibi Report sec 6 er results er jonno. 